# HighD FlowIS End-to-End Pipeline (Refactor Draft)

这个 notebook 用于可视化检查 FlowIS 的核心链路：
1. 分布拟合/加载后的场景-行为采样输入
2. 仿真器中的行为注入与逐步重要性权重
3. 最终统计图与诊断图


In [1]:
import traceback
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium.envs.registration import registry
import highway_env
warnings.filterwarnings('ignore', message='.*Overriding environment .*already in registry.*')
if 'HighDEnv-v0' not in registry and hasattr(highway_env, 'register_highway_envs'):
    highway_env.register_highway_envs()

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True
print('Imports ready')


Imports ready


## 1) 实验配置

如果你有现成的 artifact 路径，可在这里补充。


In [2]:
CONFIG = {
    'env_id': 'HighDEnv-v0',
    'episodes': 2,
    'max_steps': 200,
    'flowis_behavior_proposal': 'tde',  # or 'nde'
    'flowis_artifact_dir': None,
    'flowis_tde_path': None,
    'flowis_nde_path': None,
    # 其他外部模型/数据路径按你现有 HighD_env.ipynb 的前置单元格变量接入
}
CONFIG


{'env_id': 'HighDEnv-v0',
 'episodes': 2,
 'max_steps': 200,
 'flowis_behavior_proposal': 'tde',
 'flowis_artifact_dir': None,
 'flowis_tde_path': None,
 'flowis_nde_path': None}

## 2) 采样与注入轨迹采集

该单元会记录每一步的 info['flowis']。


In [3]:
def collect_flowis_traces(env, episodes=1, max_steps=200):
    rows = []
    for ep in range(episodes):
        obs, info = env.reset()
        for t in range(max_steps):
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            flow = info.get('flowis', {}) if isinstance(info, dict) else {}
            trace = flow.get('trace', {}) if isinstance(flow, dict) else {}
            rows.append({
                'episode': ep,
                'step': t,
                'reward': float(reward),
                'done': bool(terminated or truncated),
                'log_weight_step': flow.get('log_weight_step'),
                'log_weight_episode': flow.get('log_weight_episode'),
                'accepted': trace.get('accepted'),
                'trials': trace.get('trials'),
                'behavior': trace.get('behavior'),
                'log_q_cond': trace.get('log_q_cond'),
                'log_p_cond': trace.get('log_p_cond'),
                'scene_raw': trace.get('scene_raw'),
                'scene_behavior_raw': trace.get('scene_behavior_raw'),
                'sampler': trace.get('sampler'),
            })
            if terminated or truncated:
                break
    return pd.DataFrame(rows)

df = None
try:
    env = gym.make(CONFIG['env_id'])
    env.unwrapped.configure({
        'flowis_behavior_proposal': CONFIG['flowis_behavior_proposal'],
        'flowis_trace': True,
        'flowis_artifact_dir': CONFIG['flowis_artifact_dir'],
        'flowis_tde_path': CONFIG['flowis_tde_path'],
        'flowis_nde_path': CONFIG['flowis_nde_path'],
    })
    df = collect_flowis_traces(env, CONFIG['episodes'], CONFIG['max_steps'])
    env.close()
    print(f'Collected {len(df)} rows')
except Exception as e:
    print('Collection failed. External HighD artifacts may be missing.')
    print(type(e).__name__, e)
    traceback.print_exc(limit=1)

df.head() if isinstance(df, pd.DataFrame) and len(df) else df


Collection failed. External HighD artifacts may be missing.
FileNotFoundError FlowIS artifact not found for 'flowis_tde_path'. Tried:
 - D:\LocalSyncdisk\加速测试\FlowIS\Artifacts\FlowIS_highway.joblib
 - FlowIS_highway.joblib


Traceback (most recent call last):
  File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_22124\4002194638.py", line 32, in <module>
    env = gym.make(CONFIG['env_id'])
          ^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: FlowIS artifact not found for 'flowis_tde_path'. Tried:
 - D:\LocalSyncdisk\加速测试\FlowIS\Artifacts\FlowIS_highway.joblib
 - FlowIS_highway.joblib


## 3) 注入过程可视化


In [ ]:
if isinstance(df, pd.DataFrame) and len(df):
    show = df.copy()
    for col in ['log_weight_step', 'log_weight_episode', 'trials', 'behavior', 'log_q_cond', 'log_p_cond']:
        show[col] = pd.to_numeric(show[col], errors='coerce')

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(show.index, show['log_weight_step'])
    axes[0].set_title('Per-step log weight')
    axes[1].plot(show.index, show['log_weight_episode'])
    axes[1].set_title('Cumulative log weight')
    axes[2].plot(show.index, show['trials'])
    axes[2].set_title('Rejection trials per step')
    plt.tight_layout()
    plt.show()
    print(show['sampler'].value_counts(dropna=False))
else:
    print('No trace data yet. Configure external artifacts and rerun collection cell.')
